# EXP-003 — GR alignment, beam and particle filter

## tl;dr

One notebook compares every requested test-deployable candidate on the legal EXP-002 protocol: safe baselines, local GR matching, beam alignment, particle filtering, fixed blends, a horizon-aware blend, and a confidence-gated blend. The committed execution uses a deterministic 120-well development sample; changing `MODE` to `full` runs all 773 wells.

## Context & Methods

Only test-available information is used: future horizontal `MD, X, Y, Z, GR`, the known TVT prefix, and the paired typewell `TVT, GR`. Train-only formation columns, `Geology`, same-well lookup, and leaderboard feedback are forbidden.

Alignment operates on every eighth horizontal row and interpolates between aligned points. This keeps one notebook practical while preserving long-scale geology. Four outer cuts match EXP-002: 50%, 65%, 75%, 85%.

In [1]:
from pathlib import Path
import json, math, time
import numpy as np
import pandas as pd

MODE = 'full'  # executed validation on all 773 wells
SEED = 42
CUTS = (0.20, 0.25, 0.33, 0.50, 0.65, 0.75, 0.85)
ALLOWED = ('MD','X','Y','Z','GR')
FORBIDDEN = {'ANCC','ASTNU','ASTNL','EGFDU','EGFDL','BUDA','Geology'}

CFG = {
    'development': dict(n_wells=120, stride=8, beam_width=16, pf_particles=32, pf_seeds=3),
    'full': dict(n_wells=None, stride=8, beam_width=16, pf_particles=32, pf_seeds=3),
}[MODE]

def project_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        q=p/'competitions/public-comp/wellbore-geology-prediction'
        if (q/'datasets/train').is_dir(): return q
        if p.name=='wellbore-geology-prediction' and (p/'datasets/train').is_dir(): return p
    raise FileNotFoundError('competition root not found')

ROOT=project_root(); DATA=ROOT/'datasets'
RESULTS=ROOT/'experiments/exp_003_gr_alignment/results'; RESULTS.mkdir(parents=True,exist_ok=True)
all_files=sorted((DATA/'train').glob('*__horizontal_well.csv'))
if CFG['n_wells']:
    sample_idx=np.linspace(0,len(all_files)-1,CFG['n_wells'],dtype=int)
    WELL_FILES=[all_files[i] for i in sample_idx]
else: WELL_FILES=all_files
print(MODE, CFG, 'wells=',len(WELL_FILES))

full {'n_wells': None, 'stride': 8, 'beam_width': 16, 'pf_particles': 32, 'pf_seeds': 3} wells= 773


## Algorithms

- `gr_local`: greedy typewell match near the expected TVT trajectory.
- `gr_beam`: keeps several plausible monotone paths and penalizes implausible speed changes.
- `gr_pf`: stochastic state propagation with GR likelihood; multiple deterministic seeds provide uncertainty.

Every method starts at the final known TVT and uses the recent prefix slope only as a motion prior.

In [2]:
def smooth(a, width=7):
    return pd.Series(np.asarray(a,float)).interpolate(limit_direction='both').rolling(width,center=True,min_periods=1).mean().to_numpy()

def rmse(y,p): return float(np.sqrt(np.mean((np.asarray(y)-np.asarray(p))**2)))

def prepare(df,tw,n_visible):
    df=df[[*ALLOWED,'TVT']].copy().sort_values('MD').reset_index(drop=True)
    assert not FORBIDDEN.intersection(df.columns)
    tw=tw[['TVT','GR']].dropna().sort_values('TVT').drop_duplicates('TVT')
    tw_t=tw.TVT.to_numpy(float); tw_g=smooth(tw.GR,7)
    idx=np.unique(np.r_[np.arange(n_visible-1,len(df),CFG['stride']),len(df)-1]).astype(int)
    x=df.MD.to_numpy(float); g=smooth(df.GR,7); y=df.TVT.to_numpy(float)
    w=df.iloc[max(0,n_visible-301):n_visible]
    dx=np.diff(w.MD); dy=np.diff(w.TVT); ok=(np.abs(dx)>1e-9)&np.isfinite(dx)&np.isfinite(dy)
    slope=float(np.median(dy[ok]/dx[ok])) if ok.any() else 0.0
    return df,tw_t,tw_g,idx,x,g,y,slope

def interpolate_path(idx,path,n):
    return np.interp(np.arange(n),idx,path)

def gr_local(tw_t,tw_g,idx,x,g,start_t,slope):
    path=np.empty(len(idx)); path[0]=start_t; prev=start_t; prev_x=x[idx[0]]
    for j in range(1,len(idx)):
        expected=prev+slope*(x[idx[j]]-prev_x)
        lo=np.searchsorted(tw_t,expected-12); hi=np.searchsorted(tw_t,expected+12)
        cand=tw_t[lo:max(lo+1,hi)]; cg=tw_g[lo:max(lo+1,hi)]
        if np.isfinite(g[idx[j]]) and len(cand):
            cost=((cg-g[idx[j]])/25.)**2+((cand-expected)/5.)**2
            prev=float(cand[np.argmin(cost)])
        else: prev=float(expected)
        path[j]=prev; prev_x=x[idx[j]]
    return path

def gr_beam(tw_t,tw_g,idx,x,g,start_t,slope,width):
    states=np.array([start_t]); scores=np.array([0.]); paths=[[]]
    for j in range(1,len(idx)):
        dm=x[idx[j]]-x[idx[j-1]]; expected=states+slope*dm
        candidates=[]
        for si,(state,base,exp) in enumerate(zip(states,scores,expected)):
            center=np.searchsorted(tw_t,exp)
            for ti in range(max(0,center-5),min(len(tw_t),center+6)):
                obs=0. if not np.isfinite(g[idx[j]]) else ((tw_g[ti]-g[idx[j]])/25.)**2
                motion=((tw_t[ti]-exp)/4.)**2
                candidates.append((base+obs+motion,tw_t[ti],si))
        candidates.sort(key=lambda z:z[0])
        chosen=[]; seen=set()
        for item in candidates:
            key=round(item[1],3)
            if key not in seen: chosen.append(item); seen.add(key)
            if len(chosen)>=width: break
        new_paths=[]
        for score,state,parent in chosen: new_paths.append(paths[parent]+[state])
        scores=np.array([z[0] for z in chosen]); states=np.array([z[1] for z in chosen]); paths=new_paths
    best=int(np.argmin(scores)); return np.array([start_t]+paths[best])

def systematic_resample(weights,rng):
    positions=(rng.random()+np.arange(len(weights)))/len(weights)
    return np.searchsorted(np.cumsum(weights),positions)

def gr_pf_one(tw_t,tw_g,idx,x,g,start_t,slope,n_particles,seed):
    rng=np.random.default_rng(seed); particles=start_t+rng.normal(0,.5,n_particles); out=[start_t]; std=[.5]
    for j in range(1,len(idx)):
        dm=x[idx[j]]-x[idx[j-1]]
        particles += slope*dm+rng.normal(0,.7,n_particles)
        if np.isfinite(g[idx[j]]):
            expected_gr=np.interp(particles,tw_t,tw_g,left=tw_g[0],right=tw_g[-1])
            weights=np.exp(-.5*((expected_gr-g[idx[j]])/25.)**2); weights+=1e-12; weights/=weights.sum()
        else: weights=np.full(n_particles,1/n_particles)
        out.append(float(np.sum(particles*weights))); std.append(float(np.sqrt(np.sum((particles-out[-1])**2*weights))))
        particles=particles[systematic_resample(weights,rng)]
    return np.array(out),np.array(std)

def gr_pf(tw_t,tw_g,idx,x,g,start_t,slope):
    paths=[]; stds=[]
    for seed in range(CFG['pf_seeds']):
        p,s=gr_pf_one(tw_t,tw_g,idx,x,g,start_t,slope,CFG['pf_particles'],SEED+seed)
        paths.append(p); stds.append(s)
    return np.mean(paths,axis=0),np.sqrt(np.mean(np.square(stds),axis=0)+np.var(paths,axis=0))


## Run common validation cases

In [3]:
started=time.time(); rows=[]; failures=[]
for wi,path in enumerate(WELL_FILES,1):
    well=path.name.split('__')[0]
    try:
        raw=pd.read_csv(path); tw=pd.read_csv(DATA/'train'/f'{well}__typewell.csv')
        for cut in CUTS:
            n_visible=int(round(len(raw)*cut))
            df,tw_t,tw_g,idx,x,g,y,slope=prepare(raw,tw,n_visible)
            start=y[n_visible-1]
            local_sparse=gr_local(tw_t,tw_g,idx,x,g,start,slope)
            beam_sparse=gr_beam(tw_t,tw_g,idx,x,g,start,slope,CFG['beam_width'])
            pf_sparse,pf_std_sparse=gr_pf(tw_t,tw_g,idx,x,g,start,slope)
            target_idx=np.arange(n_visible,len(df)); sparse_x=idx
            pred_local=np.interp(target_idx,sparse_x,local_sparse)
            pred_beam=np.interp(target_idx,sparse_x,beam_sparse)
            pred_pf=np.interp(target_idx,sparse_x,pf_sparse)
            pf_std=np.interp(target_idx,sparse_x,pf_std_sparse)
            truth=y[n_visible:]
            last=np.full(len(truth),start)
            w=df.iloc[max(0,n_visible-301):n_visible]
            residual_coef=np.polyfit(w.MD,w.TVT+w.Z,1)
            zres=np.polyval(residual_coef,df.MD.iloc[n_visible:])-df.Z.iloc[n_visible:].to_numpy()
            # Confidence is target-free: lower PF dispersion and beam/PF agreement -> more trust.
            agreement=np.median(np.abs(pred_beam-pred_pf)); dispersion=np.median(pf_std)
            confidence=float(np.exp(-agreement/8.)*np.exp(-dispersion/8.))
            horizon=np.linspace(0,1,len(truth),endpoint=True)
            adaptive_weight=np.clip(confidence*(0.65-0.35*horizon),0.05,0.65)
            preds={
                'last_value':last,'z_residual_linear':zres,'gr_local':pred_local,
                'gr_beam':pred_beam,'gr_pf':pred_pf,
                'last_beam_w25':.75*last+.25*pred_beam,
                'last_beam_w50':.50*last+.50*pred_beam,
                'last_beam_w75':.25*last+.75*pred_beam,
                'last_pf_w50':.50*last+.50*pred_pf,
                'beam_pf_w50':.50*pred_beam+.50*pred_pf,
                'horizon_blend':(1-adaptive_weight)*last+adaptive_weight*.5*(pred_beam+pred_pf),
            }
            for model,pred in preds.items():
                err=np.asarray(pred)-truth
                rows.append(dict(well=well,cut_fraction=cut,model=model,n_hidden=len(truth),rmse=rmse(truth,pred),bias=float(err.mean()),squared_error_sum=float(np.sum(err**2)),confidence=confidence,beam_pf_agreement=agreement,pf_dispersion=dispersion))
    except Exception as exc: failures.append(dict(well=well,reason=repr(exc)))
    if wi%20==0: print(f'{wi}/{len(WELL_FILES)} wells, {time.time()-started:.0f}s')
results=pd.DataFrame(rows); failures_df=pd.DataFrame(failures); elapsed=time.time()-started
print('done',results.well.nunique(),'wells',len(results),'metrics','failures',len(failures_df),'seconds',elapsed)
assert results.well.nunique() >= len(WELL_FILES)-len(failures_df)
results.head()

20/773 wells, 16s


40/773 wells, 34s


60/773 wells, 52s


80/773 wells, 71s


100/773 wells, 90s


...

720/773 wells, 634s


740/773 wells, 651s


760/773 wells, 669s


done 773 wells 59521 metrics failures 0 seconds 679.2054979801178


,well,cut_fraction,model,n_hidden,rmse,bias,squared_error_sum,confidence,beam_pf_agreement,pf_dispersion
0,000d7d20,0.2,last_value,4222,7.405365,5.425438,2.315321e+05,0.002866,42.758934,4.080128
1,000d7d20,0.2,z_residual_linear,4222,14.369297,-12.021900,8.717446e+05,0.002866,42.758934,4.080128
2,000d7d20,0.2,gr_local,4222,17.699276,-15.810612,1.322602e+06,0.002866,42.758934,4.080128
3,000d7d20,0.2,gr_beam,4222,8.471894,-3.474693,3.030256e+05,0.002866,42.758934,4.080128
4,000d7d20,0.2,gr_pf,4222,41.477371,-37.747129,7.263412e+06,0.002866,42.758934,4.080128


## Ablation results

In [4]:
def summarize(g):
    return pd.Series(dict(
        pooled_rmse=np.sqrt(g.squared_error_sum.sum()/g.n_hidden.sum()),
        mean_well_rmse=g.rmse.mean(),median_well_rmse=g.rmse.median(),
        p90_well_rmse=g.rmse.quantile(.9),worst_well_rmse=g.rmse.max(),
        well_cases=len(g),hidden_rows=g.n_hidden.sum()))
summary=results.groupby('model',sort=False).apply(summarize,include_groups=False).reset_index().sort_values('pooled_rmse')
by_cut=results.groupby(['cut_fraction','model'],sort=False).apply(summarize,include_groups=False).reset_index()
winners=results.loc[results.groupby(['well','cut_fraction']).rmse.idxmin()].groupby('model').size().rename('wins').reset_index().sort_values('wins',ascending=False)
display(summary.round(4)); display(by_cut.pivot(index='model',columns='cut_fraction',values='pooled_rmse').round(4)); display(winners)

,model,pooled_rmse,mean_well_rmse,median_well_rmse,p90_well_rmse,worst_well_rmse,well_cases,hidden_rows
5,last_beam_w25,16.0657,11.2299,8.7905,21.2293,215.7213,5411.0,17670102.0
10,horizon_blend,16.7063,11.2237,8.7732,21.0392,307.6202,5411.0,17670102.0
0,last_value,18.0477,11.4402,8.7286,20.9474,346.7590,5411.0,17670102.0
6,last_beam_w50,19.6011,12.7074,9.3943,24.8330,136.7034,5411.0,17670102.0
7,last_beam_w75,26.5333,15.2014,10.1413,28.5314,198.8124,5411.0,17670102.0
2,gr_local,28.0088,16.1509,10.4854,33.9764,206.1437,5411.0,17670102.0
3,gr_beam,34.8933,18.1605,11.2242,33.0986,270.8757,5411.0,17670102.0
1,z_residual_linear,43.0080,22.0096,12.7146,50.3632,514.7680,5411.0,17670102.0
8,last_pf_w50,106.2380,31.8581,10.8006,36.0221,852.1596,5411.0,17670102.0
9,beam_pf_w50,122.0928,36.8455,11.8435,43.9030,936.3686,5411.0,17670102.0


cut_fraction,0.20,0.25,0.33,0.50,0.65,0.75,0.85
model,,,,,,,
beam_pf_w50,238.5432,84.5338,27.4989,19.5914,15.7760,13.1979,9.9906
gr_beam,61.1096,30.1434,20.0497,17.6253,14.4745,13.0841,10.0355
gr_local,45.9053,26.5552,19.4095,16.5361,14.0779,11.8082,9.7012
gr_pf,426.5834,146.1573,39.5590,25.1865,19.9707,15.5908,11.6064
horizon_blend,23.6373,16.8505,14.6809,12.6780,11.5076,10.3927,8.3387
last_beam_w25,21.8642,16.3299,14.6537,12.9906,11.6003,10.4795,8.3246
last_beam_w50,29.8697,18.7818,15.6406,14.0295,12.1381,10.9587,8.5475
last_beam_w75,44.4707,23.8064,17.5228,15.6245,13.1268,11.8554,9.1395
last_pf_w50,208.1089,72.2576,22.8889,16.4672,13.9611,11.5666,9.1226


,model,wins
10,z_residual_linear,1250
9,last_value,1052
2,gr_local,571
1,gr_beam,504
3,gr_pf,456
5,last_beam_w25,321
4,horizon_blend,274
8,last_pf_w50,272
6,last_beam_w50,264
0,beam_pf_w50,228


## Confidence diagnostics

Confidence is useful only if lower confidence corresponds to larger alignment error. The table checks that relationship without tuning thresholds on the public leaderboard.

In [5]:
beam=results[results.model=='gr_beam'].copy(); beam['confidence_bin']=pd.qcut(beam.confidence,4,duplicates='drop')
confidence_report=beam.groupby('confidence_bin',observed=True).agg(cases=('rmse','size'),mean_confidence=('confidence','mean'),beam_rmse=('rmse','mean'),p90_rmse=('rmse',lambda x:x.quantile(.9))).reset_index()
display(confidence_report)

,confidence_bin,cases,mean_confidence,beam_rmse,p90_rmse
0,"(-0.001, 0.0377]",1353,0.009865,36.065198,107.740960
1,"(0.0377, 0.158]",1353,0.091424,14.309719,26.345645
2,"(0.158, 0.323]",1352,0.238203,12.417328,24.744748
3,"(0.323, 0.889]",1353,0.461004,9.845471,21.277196


## Takeaways

Promotion rule: a candidate must beat the sample-matched `last_value` pooled RMSE and avoid material p90 degradation. If GR methods fail, EXP-004 should not increase PF complexity; it should move to supervised delta prediction or a more faithful sequence model.

In [6]:
results.to_csv(RESULTS/'per_well_cut_metrics.csv',index=False); summary.to_csv(RESULTS/'summary.csv',index=False); by_cut.to_csv(RESULTS/'summary_by_cut.csv',index=False); winners.to_csv(RESULTS/'winner_counts.csv',index=False); confidence_report.to_csv(RESULTS/'confidence_report.csv',index=False); failures_df.to_csv(RESULTS/'failures.csv',index=False)
run=dict(experiment_id='exp_003',mode=MODE,config=CFG,wells=int(results.well.nunique()),metrics=int(len(results)),elapsed_sec=elapsed,best_model=str(summary.iloc[0].model),best_pooled_rmse=float(summary.iloc[0].pooled_rmse),anchor_pooled_rmse=float(summary.loc[summary.model=='last_value','pooled_rmse'].iloc[0]))
(RESULTS/'run.json').write_text(json.dumps(run,indent=2)+'\n'); print(json.dumps(run,indent=2)); print('saved',sorted(p.name for p in RESULTS.iterdir()))

{
  "experiment_id": "exp_003",
  "mode": "full",
  "config": {
    "n_wells": null,
    "stride": 8,
    "beam_width": 16,
    "pf_particles": 32,
    "pf_seeds": 3
  },
  "wells": 773,
  "metrics": 59521,
  "elapsed_sec": 679.2054979801178,
  "best_model": "last_beam_w25",
  "best_pooled_rmse": 16.065684045482225,
  "anchor_pooled_rmse": 18.04774980272698
}
saved ['confidence_report.csv', 'failures.csv', 'per_well_cut_metrics.csv', 'run.json', 'summary.csv', 'summary_by_cut.csv', 'winner_counts.csv']
